In [ ]:
import os, glob
import numpy as np
import xarray as xr
from dask.diagnostics import ProgressBar

# ============================================================
# BlockB2 — Zonal wind (U) zonal-mean -> std pressure levels
# Output: (time, plev, lat) for ALL Latitudes (Global)
# ============================================================

# ---------------- paths ----------------
OUT_DIR = "/home/weiji/restart_exam/code/20260122StructureAnalysis/result"
os.makedirs(OUT_DIR, exist_ok=True)

BWCN_H3_DIR = "/mnt/backup_ETH/EXTR_2000/EXTR_2000/BWCN.e122.f19_g16.002/atm/hist"

OUT_U_NC = os.path.join(OUT_DIR, "U_BWCN002_zm_ALL_LAT_stdplev_time_plev_lat.nc")

# 标准 pressure levels (Pa)
PLEV_STD_PA = np.array([
    10, 50, 100, 200, 300, 500, 1000, 2000, 3000, 5000, 7000, 10000, 15000,
    20000, 25000, 30000, 40000, 50000, 60000, 70000, 85000, 92500, 100000
], dtype=float)

# dask chunks（time=1 比较稳）
CHUNKS = {"time": 1}

# ---------------- helpers ----------------
def compute_pressure_mid(ds: xr.Dataset) -> xr.DataArray:
    """p_mid = hyam*P0 + hybm*PS  (Pa)"""
    hyam = ds["hyam"]   # (lev)
    hybm = ds["hybm"]   # (lev)
    P0   = ds["P0"]     # scalar Pa
    PS   = ds["PS"]     # (time, lat, lon) Pa
    p_mid = hyam * P0 + hybm * PS
    p_mid.name = "p_mid"
    p_mid.attrs["units"] = "Pa"
    return p_mid

def interp_profile_logp(v_hyb: xr.DataArray, p_hyb: xr.DataArray, p_tgt: np.ndarray) -> xr.DataArray:
    """
    v_hyb, p_hyb: (..., lev) 其中 p_hyb 单位 Pa
    p_tgt: (nplev,) 目标标准气压层 Pa
    return: (..., plev) log(p) 线性插值
    """
    p_tgt = np.asarray(p_tgt, float)

    def _interp_col(vcol, pcol):
        m = np.isfinite(vcol) & np.isfinite(pcol) & (pcol > 0)
        if m.sum() < 2:
            return np.full(p_tgt.shape, np.nan, float)

        p_use = pcol[m]
        v_use = vcol[m]

        # np.interp 要求自变量递增：按 pressure 升序
        idx = np.argsort(p_use)
        p_use = p_use[idx]
        v_use = v_use[idx]

        return np.interp(np.log(p_tgt), np.log(p_use), v_use, left=np.nan, right=np.nan)

    out = xr.apply_ufunc(
        _interp_col,
        v_hyb, p_hyb,
        input_core_dims=[["lev"], ["lev"]],
        output_core_dims=[["plev"]],
        vectorize=True,
        dask="parallelized",
        output_dtypes=[float],
        output_sizes={"plev": len(p_tgt)},
    )

    out = out.assign_coords(plev=("plev", p_tgt))
    out["plev"].attrs["units"] = "Pa"
    return out

# ---------------- main ----------------
h3_files = sorted(glob.glob(os.path.join(BWCN_H3_DIR, "*.cam.h3.*.nc")))
if len(h3_files) == 0:
    raise RuntimeError(f"No h3 files found in: {BWCN_H3_DIR}")

print(f"[BlockB2_U] Found {len(h3_files)} h3 files.")
print("[BlockB2_U] Opening multi-file dataset...")

ds = xr.open_mfdataset(
    h3_files,
    combine="by_coords",
    parallel=True,
    use_cftime=True,
    chunks=CHUNKS,
    data_vars="minimal",
    coords="minimal",
    compat="override",
)

need_vars = ["U", "PS", "P0", "hyam", "hybm"]
for v in need_vars:
    if v not in ds:
        raise RuntimeError(f"[BlockB2_U] Missing variable '{v}' in BWCN h3 files.")

U = ds["U"]   # (time, lev, lat, lon) usually m/s in CAM

# 清理超大缺测值
U = U.where(np.abs(U) < 1e20)

# ---- zonal mean ----
U_zm = U.mean("lon")  # (time, lev, lat)
U_zm.name = "U_zm_hybrid"
U_zm.attrs["long_name"] = "Zonal mean zonal wind U"
# 尽量保留原 units；没有的话就补一个常见的
U_zm.attrs["units"] = U.attrs.get("units", "m s-1")

# ---- pressure (zonal mean) for vertical interpolation ----
p_mid = compute_pressure_mid(ds)   # (time, lev, lat, lon)
p_zm  = p_mid.mean("lon")          # (time, lev, lat)

# ---- interpolate U to standard pressure levels ----
print("[BlockB2_U] Interpolating U(zm) to standard pressure levels (log-p)...")
U_std = interp_profile_logp(U_zm, p_zm, PLEV_STD_PA)  # (time, lat, plev)

# 调整维度顺序 (time, plev, lat)
U_std = U_std.transpose("time", "plev", "lat")

U_std.name = "U_zm_stdplev"
U_std.attrs.update(
    units=U_zm.attrs.get("units", "m s-1"),
    long_name="Zonal mean zonal wind U interpolated to standard pressure levels",
    interp_method="log(p) linear interpolation using zonal-mean mid-level pressure",
)

print(f"[BlockB2_U] Output shape: {U_std.shape} (time, plev, lat)")

# ---- write to netcdf (compressed) ----
encoding = {
    "U_zm_stdplev": {
        "zlib": True,
        "complevel": 4,
        "dtype": "float32",
        "_FillValue": np.float32(np.nan),
    },
    "plev": {"dtype": "float64"},
}

print(f"[BlockB2_U] Saving to {OUT_U_NC} ...")

# 真正触发计算并写入文件
# 使用 ProgressBar 可以看到进度条
with ProgressBar():
    U_std.to_netcdf(OUT_U_NC, format="NETCDF4", encoding=encoding)

print("[BlockB2_U] Done.")
